# Seasonal Agriculture Performance Analysis
**VOIS AICTE Major Project | Data Analytics**

## Project Goal
Analyze agricultural performance across **Kharif, Rabi and Zaid** seasons, identify patterns in yield, production, profitability, resource use and environmental conditions, and translate the evidence into practical recommendations.

### Analytical questions
1. How does yield vary by season?
2. Which season is most profitable?
3. How do cost, revenue and profit change across seasons?
4. How does water use and water efficiency vary?
5. What environmental/resource variables are associated with yield?
6. Which crop-season combinations appear financially attractive?
7. Are seasonal yield differences statistically meaningful?

**Data source:** `seasonal_agriculture_performance_dataset.csv`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
df = pd.read_csv('seasonal_agriculture_performance_dataset.csv')
print('Shape:', df.shape)
display(df.head())


## 1. Data Quality & Preparation

In [ ]:
quality = pd.DataFrame({
    'Missing Values': df.isna().sum(),
    'Missing %': (df.isna().mean()*100).round(2),
    'Unique Values': df.nunique()
}).sort_values('Missing Values', ascending=False)
display(quality[quality['Missing Values']>0])

print('Duplicate rows:', df.duplicated().sum())

# Median imputation for numeric fields with missing values
clean = df.copy()
numeric_impute = ['Rainfall_mm','Soil_Moisture_pct','Yield_Tonnes_Ha']
for col in numeric_impute:
    clean[col] = clean[col].fillna(clean[col].median())

print('Remaining missing values:', int(clean.isna().sum().sum()))


## 2. Dataset Overview

In [ ]:
print(clean.describe(include='all').T)
print('\nSeason distribution:')
display(clean['Season'].value_counts().rename_axis('Season').to_frame('Farms'))


## 3. Seasonal Performance

In [ ]:
season = clean.groupby('Season').agg(
    Farms=('Farm_ID','count'),
    Avg_Yield=('Yield_Tonnes_Ha','mean'),
    Avg_Production=('Production_Tonnes','mean'),
    Avg_Revenue=('Revenue_INR','mean'),
    Avg_Cost=('Total_Cost_INR','mean'),
    Avg_Profit=('Profit_INR','mean'),
    Avg_Water=('Water_Used_m3','mean'),
    Avg_Water_Efficiency=('Water_Efficiency_t_per_1000m3','mean'),
    Avg_Disease_Risk=('Disease_Pest_Risk_pct','mean')
).sort_values('Avg_Profit', ascending=False)
season['Profit_Margin_%'] = season['Avg_Profit']/season['Avg_Revenue']*100
display(season.round(2))


In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
season['Avg_Yield'].plot(kind='bar', ax=ax)
ax.set_title('Average Yield by Season')
ax.set_ylabel('Tonnes / Hectare')
ax.set_xlabel('Season')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
season[['Avg_Revenue','Avg_Cost','Avg_Profit']].plot(kind='bar', ax=ax)
ax.set_title('Average Revenue, Cost and Profit by Season')
ax.set_ylabel('INR per farm')
ax.set_xlabel('Season')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
season[['Avg_Water','Avg_Water_Efficiency']].plot(kind='bar', ax=ax)
ax.set_title('Water Use and Water Efficiency by Season')
ax.set_ylabel('Average value')
ax.set_xlabel('Season')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 4. Crop × Season Analysis

In [ ]:
crop_season = clean.groupby(['Season','Crop']).agg(
    Farms=('Farm_ID','count'),
    Avg_Yield=('Yield_Tonnes_Ha','mean'),
    Avg_Revenue=('Revenue_INR','mean'),
    Avg_Cost=('Total_Cost_INR','mean'),
    Avg_Profit=('Profit_INR','mean')
).reset_index()
crop_season['Margin_%'] = crop_season['Avg_Profit']/crop_season['Avg_Revenue']*100
display(crop_season.sort_values('Margin_%', ascending=False).head(15).round(2))


## 5. Relationships with Yield

In [ ]:
corr_cols = ['Yield_Tonnes_Ha','Rainfall_mm','Avg_Temperature_C','Humidity_pct',
              'Sunlight_Hours_Day','Soil_pH','Soil_Moisture_pct','Nitrogen_kg_ha',
              'Phosphorus_kg_ha','Potassium_kg_ha','Fertilizer_kg_ha',
              'Pesticide_Litre_ha','Water_Used_m3','Disease_Pest_Risk_pct']
corr = clean[corr_cols].corr(numeric_only=True)['Yield_Tonnes_Ha'].sort_values(ascending=False)
display(corr.to_frame('Correlation with Yield'))


In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
corr.drop('Yield_Tonnes_Ha').plot(kind='bar', ax=ax)
ax.set_title('Correlation of Factors with Yield')
ax.set_ylabel('Pearson correlation')
ax.set_xlabel('Variable')
plt.xticks(rotation=70, ha='right')
plt.tight_layout()
plt.show()


## 6. Statistical Test: Seasonal Yield Differences

In [ ]:
groups = [g['Yield_Tonnes_Ha'].dropna().values for _, g in clean.groupby('Season')]
anova = stats.f_oneway(*groups)
print(f'One-way ANOVA F-statistic: {anova.statistic:.3f}')
print(f'p-value: {anova.pvalue:.6g}')
if anova.pvalue < 0.05:
    print('Conclusion: Mean yield differs significantly across at least one season at the 5% level.')
else:
    print('Conclusion: No statistically significant seasonal difference detected at the 5% level.')


## 7. Key Findings & Recommendations

The analysis should be interpreted from the computed tables and charts rather than from assumptions. For this dataset, the strongest decision signals are:
- compare seasons using yield **and** profitability, not yield alone;
- monitor Zaid economics because its average cost exceeds average revenue in the supplied data;
- evaluate crop-season combinations because overall seasonal averages can hide profitable niches;
- use water efficiency alongside total water consumption;
- investigate environmental drivers that show stronger correlations with yield;
- use statistically significant seasonal differences as evidence for seasonal planning, not as proof of causation.

### Recommended business actions
1. Prioritize financially resilient crop-season combinations.
2. Review cost structure and irrigation strategy for loss-making seasonal segments.
3. Track water efficiency as a core KPI.
4. Strengthen disease/pest monitoring where risk is elevated.
5. Repeat the analysis by district/state to identify local exceptions.
